In [2]:
# Load the cleaned dataset produced in Phase 1
import pandas as pd

# parse_dates converts Date to datetime, index_col makes it the index
df = pd.read_csv(
    "../data/cleaned_data/google_stock_price_cleaned.csv",
    parse_dates=["Date"],
    index_col="Date",
)

print(df.shape)
df.head()

(1939, 6)


,Open,High,Close,Low,Adj Close,Volume
Date,,,,,,
2019-01-02,50.828499,52.616001,52.292500,50.785500,51.801849,30652000
2019-01-03,52.049999,52.848999,50.803001,50.703499,50.326332,36822000
2019-01-04,51.629501,53.542000,53.535500,51.370899,53.033192,41878000
2019-01-07,53.575001,53.700001,53.419498,52.737999,52.918270,39638000
2019-01-08,53.805500,54.228001,53.813999,53.026501,53.309071,35298000


In [3]:
# Select the columns the model will use as input
# Adj Close is dropped: it is nearly identical to Close
# (it only differs by a small dividend adjustment)
features = ["Open", "High", "Low", "Close", "Volume"]
df = df[features]

print(df.shape)
df.head()

(1939, 5)


,Open,High,Low,Close,Volume
Date,,,,,
2019-01-02,50.828499,52.616001,50.785500,52.292500,30652000
2019-01-03,52.049999,52.848999,50.703499,50.803001,36822000
2019-01-04,51.629501,53.542000,51.370899,53.535500,41878000
2019-01-07,53.575001,53.700001,52.737999,53.419498,39638000
2019-01-08,53.805500,54.228001,53.026501,53.813999,35298000


In [4]:
# Chronological split (no shuffling, to avoid data leakage from the future)
train = df.loc["2019-01-01":"2024-12-31"]
val = df.loc["2025-01-01":"2025-12-31"]
test = df.loc["2026-01-01":]

for name, part in [("Train", train), ("Validation", val), ("Test", test)]:
    print(f"{name}: {len(part)} rows, {part.index.min().date()} to {part.index.max().date()}")

print("Total:", len(train) + len(val) + len(test))

Train: 1510 rows, 2019-01-02 to 2024-12-31
Validation: 250 rows, 2025-01-02 to 2025-12-31
Test: 179 rows, 2026-01-02 to 2026-09-18
Total: 1939


In [5]:
from sklearn.preprocessing import MinMaxScaler
import joblib

# Fit the scaler on the training data only, to avoid data leakage
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train)

# Validation and test are only transformed with the training min/max
val_scaled = scaler.transform(val)
test_scaled = scaler.transform(test)

# Save the scaler, it is needed later to convert predictions back to real prices
joblib.dump(scaler, "../saved_models/scaler.pkl")

print("Train  min/max:", train_scaled.min().round(3), train_scaled.max().round(3))
print("Val    min/max:", val_scaled.min().round(3), val_scaled.max().round(3))
print("Test   min/max:", test_scaled.min().round(3), test_scaled.max().round(3))

Train  min/max: 0.0 1.0
Val    min/max: -0.006 1.868
Test   min/max: 0.028 2.363


In [6]:
import numpy as np

# Number of past days the model sees for each prediction
LOOKBACK = 60

# The target is the Close price (its column position among the features)
target_idx = features.index("Close")

def make_sequences(data, lookback=LOOKBACK):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i])   # previous 60 days, all features
        y.append(data[i, target_idx])    # Close price of the next day
    return np.array(X), np.array(y)

# Validation/test get the last 60 rows of the previous split as context
# so that their first prediction also has a full 60 day window
X_train, y_train = make_sequences(train_scaled)
X_val, y_val = make_sequences(np.vstack([train_scaled[-LOOKBACK:], val_scaled]))
X_test, y_test = make_sequences(np.vstack([val_scaled[-LOOKBACK:], test_scaled]))

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

X_train: (1450, 60, 5) y_train: (1450,)
X_val:   (250, 60, 5) y_val:   (250,)
X_test:  (179, 60, 5) y_test:  (179,)


In [7]:
# Dates of the day each target (y) belongs to
# Needed later to plot predictions against real dates
train_dates = train.index[LOOKBACK:]
val_dates = val.index
test_dates = test.index

# Save arrays in compressed .npz files, one per split
np.savez_compressed("../data/model_ready_data/train.npz", X=X_train, y=y_train, dates=train_dates.values)
np.savez_compressed("../data/model_ready_data/val.npz", X=X_val, y=y_val, dates=val_dates.values)
np.savez_compressed("../data/model_ready_data/test.npz", X=X_test, y=y_test, dates=test_dates.values)

# Quick check: reload one file and compare shapes
check = np.load("../data/model_ready_data/train.npz")
print("Reloaded X:", check["X"].shape, "y:", check["y"].shape, "dates:", check["dates"].shape)

Reloaded X: (1450, 60, 5) y: (1450,) dates: (1450,)


# Data Preprocessing Pipeline

```text
Raw Data
   ↓
Feature Selection
   ↓
Train / Validation / Test Split
   ↓
MinMax Scaling
   ↓
60-day Sequences
   ↓
X + y + Target Dates
   ↓
train.npz / val.npz / test.npz
   ↓
Model Training